# Modeling Readiness Assessor — Provisioner Teardown

Removes resources created by provisioner.ipynb:
- Semantic models: CRM-Sales, ERP-Finance, Invoicing-Legacy
- Fabric IQ ontology: Manufacturing-Ontology

**Safety gate**: Requires `demo_workspace: true` in narrator.config.yaml.
**Idempotent**: Skips resources that no longer exist.

In [ ]:
import yaml, sys
from pathlib import Path

CONFIG_PATH = Path('/lakehouse/default/Files/narrator.config.yaml')
if not CONFIG_PATH.exists():
    CONFIG_PATH = Path('narrator.config.yaml')

config = yaml.safe_load(CONFIG_PATH.read_text()) if CONFIG_PATH.exists() else {}

if not config.get('demo_workspace', False):
    raise RuntimeError(
        "SAFETY GATE: demo_workspace is not set to true in narrator.config.yaml. "
        "Teardown aborted — no resources were deleted."
    )

print('✓ demo_workspace: true confirmed — teardown may proceed')

In [ ]:
confirm = input("Type 'yes' to DELETE demo resources from this workspace, or anything else to abort: ")
if confirm.strip().lower() != 'yes':
    print('Aborted — no resources were deleted.')
    sys.exit(0)
print('✓ User confirmed — beginning teardown...')

In [ ]:
import requests

def get_token(scope='pbi'):
    return notebookutils.credentials.getToken(scope)  # noqa: F821

def get_workspace_id():
    return notebookutils.runtime.context['currentWorkspaceId']  # noqa: F821

PBI_API = 'https://api.powerbi.com/v1.0/myorg'
DEMO_MODEL_NAMES = {'CRM-Sales', 'ERP-Finance', 'Invoicing-Legacy'}

workspace_id = get_workspace_id()
token = get_token('pbi')
headers = {'Authorization': f'Bearer {token}'}

resp = requests.get(f'{PBI_API}/groups/{workspace_id}/datasets', headers=headers)
all_models = resp.json().get('value', [])
to_delete = [m for m in all_models if m['name'] in DEMO_MODEL_NAMES]

deleted = []
for model in to_delete:
    del_resp = requests.delete(
        f'{PBI_API}/groups/{workspace_id}/datasets/{model["id"]}',
        headers=headers,
    )
    if del_resp.status_code in (200, 204):
        print(f'  ✓ {model["name"]}: deleted')
        deleted.append(model['name'])
    else:
        print(f'  ✗ {model["name"]}: delete failed ({del_resp.status_code})')

skipped = DEMO_MODEL_NAMES - {m['name'] for m in to_delete}
for name in skipped:
    print(f'  – {name}: not found, skipping')

print(f'Semantic models: {len(deleted)} deleted, {len(skipped)} not found')

In [ ]:
FABRIC_API = 'https://api.fabric.microsoft.com/v1'
ONTOLOGY_NAME = 'Manufacturing-Ontology'

onto_token = get_token('fabric')
onto_headers = {'Authorization': f'Bearer {onto_token}'}

existing = requests.get(
    f'{FABRIC_API}/workspaces/{workspace_id}/ontologies',
    headers=onto_headers,
)
ontologies = existing.json().get('value', [])
target = next((o for o in ontologies if o['name'] == ONTOLOGY_NAME), None)

if target:
    del_resp = requests.delete(
        f'{FABRIC_API}/workspaces/{workspace_id}/ontologies/{target["id"]}',
        headers=onto_headers,
    )
    if del_resp.status_code in (200, 204):
        print(f'  ✓ {ONTOLOGY_NAME}: deleted')
    else:
        print(f'  ✗ {ONTOLOGY_NAME}: delete failed ({del_resp.status_code})')
else:
    print(f'  – {ONTOLOGY_NAME}: not found, skipping')

In [ ]:
print()
print('=== Teardown Summary ===')
print(f'Workspace: {workspace_id}')
print()
print('Demo resources removed (or already absent):')
print('  Semantic models: CRM-Sales, ERP-Finance, Invoicing-Legacy')
print('  Ontology: Manufacturing-Ontology')
print()
print('Note: OneLake scan artifacts (Files/modeling-readiness/) are NOT removed.')
print('Delete those manually if needed.')